# Transfer learning with a pretrained ResNet-18

Fine-tune an ImageNet-pretrained `resnet18` on your own images (or CIFAR-10) with PyTorch.

**What this notebook covers**

1. Data loading + augmentation
2. Loading pretrained weights and replacing the classifier head
3. Two-stage training: frozen backbone (feature extraction) → optional fine-tuning
4. Loss/accuracy curves, confusion matrix, and sample predictions
5. Checkpoint save/load, 512-d feature extraction, and a flattened weight vector

**Requirements:** `torch`, `torchvision`, `matplotlib`, `scikit-learn`

In [ ]:
# If something is missing:
# %pip install torch torchvision matplotlib scikit-learn

## 1. Setup and configuration

`DATA_DIR` is used in two ways:

* If it contains **one subfolder per class**, it is loaded with `ImageFolder`:
  ```
  data/
  ├── cats/   img_001.jpg ...
  ├── dogs/   img_001.jpg ...
  └── ...
  ```
* Otherwise **CIFAR-10** is downloaded there automatically.

On CPU, start with `IMG_SIZE = 128` and `TRAIN_FRACTION = 0.2` to check that everything runs.

In [ ]:
import os, random, time
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms, models
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, classification_report

# ----------------------------- config -----------------------------
SEED            = 42
DATA_DIR        = Path("../data")   # ImageFolder root (class subfolders) or CIFAR-10 download dir
IMG_SIZE        = 128               # 224 = native ResNet size; 128 is ~3x faster
BATCH_SIZE      = 64
NUM_WORKERS     = 0                 # Windows/Jupyter: keep 0
VAL_SPLIT       = 0.02
TRAIN_FRACTION  = 0.1               # e.g. 0.2 for a quick smoke test
EPOCHS          = 2
LR_HEAD         = 1e-3              # LR for the new classifier head
LR_BACKBONE     = 1e-4              # LR for the pretrained backbone (once unfrozen)
WEIGHT_DECAY    = 1e-4
LABEL_SMOOTHING = 0.1
FREEZE_BACKBONE = True              # start as a pure feature extractor
UNFREEZE_AT     = 1                 # 0-based epoch to unfreeze; None = never unfreeze

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.backends.cudnn.benchmark = device.type == "cuda"

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)
print("device:", device)

## 2. Data and augmentation

Pretrained ImageNet models expect **224×224** inputs normalised with the ImageNet mean/std, so we resize/upscale and use those statistics even for CIFAR-10.

* Training: random crop, flip, slight colour jitter.
* Validation/test: resize + centre crop only.

In [ ]:
MEAN = [0.485, 0.456, 0.406]
STD  = [0.229, 0.224, 0.225]

train_tf = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.65, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(0.2, 0.2, 0.2),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

eval_tf = transforms.Compose([
    transforms.Resize(int(IMG_SIZE * 1.14)),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

def find_imagefolder(root):
    '''Return root if it directly contains class subfolders with images.'''
    root = Path(root)
    if not root.is_dir():
        return None
    subdirs = [p for p in root.iterdir() if p.is_dir()]
    if not subdirs:
        return None
    exts = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
    has_images = any(f.suffix.lower() in exts for sub in subdirs for f in sub.rglob("*"))
    return root if has_images else None

folder = find_imagefolder(DATA_DIR)

if folder is not None:
    # --- your own data: one subfolder per class ---
    train_full = datasets.ImageFolder(folder, transform=train_tf)
    eval_full  = datasets.ImageFolder(folder, transform=eval_tf)
    classes = train_full.classes
    test_ds = None
    print(f"ImageFolder: {len(train_full)} images, {len(classes)} classes")
else:
    # --- fallback: CIFAR-10 (downloaded automatically) ---
    train_full = datasets.CIFAR10(DATA_DIR, train=True,  download=True, transform=train_tf)
    eval_full  = datasets.CIFAR10(DATA_DIR, train=True,  download=True, transform=eval_tf)
    classes = train_full.classes
    test_ds = datasets.CIFAR10(DATA_DIR, train=False, download=True, transform=eval_tf)
    print(f"CIFAR-10: {len(train_full)} train images, {len(classes)} classes")

# random split: the same indices are used for the train and eval transforms
n = len(train_full)
n_val = max(1, int(n * VAL_SPLIT))
g = torch.Generator().manual_seed(SEED)
perm = torch.randperm(n, generator=g).tolist()
val_idx, train_idx = perm[:n_val], perm[n_val:]

# optional quick smoke test
if TRAIN_FRACTION < 1.0:
    train_idx = train_idx[: max(1, int(len(train_idx) * TRAIN_FRACTION))]

if not train_idx:
    raise ValueError("Not enough images for a train/val split.")

train_ds = Subset(train_full, train_idx)
val_ds   = Subset(eval_full,  val_idx)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=(device.type == "cuda"),
                          persistent_workers=(NUM_WORKERS > 0))
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=(device.type == "cuda"))
test_loader  = (DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False,
                           num_workers=NUM_WORKERS, pin_memory=(device.type == "cuda"))
                if test_ds is not None else None)

print(f"train: {len(train_ds)} | val: {len(val_ds)} | "
      f"test: {len(test_ds) if test_ds is not None else 'n/a'}")

In [ ]:
def denormalize(t):
    t = t.clone()
    for c, (m, s) in enumerate(zip(MEAN, STD)):
        t[c] = t[c] * s + m
    return t.clamp(0, 1)

images, labels = next(iter(train_loader))
images, labels = images[:16], labels[:16]

fig, axes = plt.subplots(2, 8, figsize=(16, 4.6))
for ax, img, lab in zip(axes.ravel(), images, labels):
    ax.imshow(denormalize(img).permute(1, 2, 0))
    ax.set_title(classes[lab], fontsize=9)
    ax.axis("off")
fig.suptitle("Training samples (with augmentation)", y=1.02)
plt.tight_layout()
plt.show()

## 3. Model: pretrained ResNet-18

`models.resnet18(weights=...)` downloads the ImageNet weights. We replace the final fully-connected layer (`model.fc`, 512 → 1000) with a new one for our number of classes.

With `FREEZE_BACKBONE = True` only the new head is trained at first. This is fast and hard to overfit, and is usually a strong baseline on small datasets.

In [ ]:
def build_model(num_classes, freeze_backbone=True):
    '''ResNet-18 pretrained on ImageNet, with a fresh classifier head.'''
    model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    if freeze_backbone:
        for name, p in model.named_parameters():
            p.requires_grad = name.startswith("fc.")
    return model

model = build_model(len(classes), freeze_backbone=FREEZE_BACKBONE).to(device)

total = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"parameters: {total/1e6:.2f}M total | {trainable/1e6:.2f}M trainable")

## 4. Loss, optimiser and training loop

A few details that matter for transfer learning:

* **Discriminative learning rates** — the new head gets `LR_HEAD`, the pretrained backbone gets the smaller `LR_BACKBONE`.
* **BatchNorm while frozen** — the backbone stays in `eval()` mode so its ImageNet running statistics are not overwritten by your (small) batches. After unfreezing, normal training mode is used.
* **AMP** — mixed precision is enabled automatically on CUDA.

In [ ]:
criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)

def make_optimizer(model):
    head = [p for n, p in model.named_parameters() if n.startswith("fc.")]
    backbone = [p for n, p in model.named_parameters()
                if not n.startswith("fc.") and p.requires_grad]
    groups = [{"params": head, "lr": LR_HEAD}]
    if backbone:
        groups.append({"params": backbone, "lr": LR_BACKBONE})
    return torch.optim.AdamW(groups, weight_decay=WEIGHT_DECAY)

optimizer = make_optimizer(model)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

use_amp = device.type == "cuda"
try:
    scaler = torch.amp.GradScaler(device.type, enabled=use_amp)
except (AttributeError, TypeError):  # older PyTorch
    scaler = torch.cuda.amp.GradScaler(enabled=use_amp)

In [ ]:
def set_backbone_trainable(model, trainable):
    for name, p in model.named_parameters():
        p.requires_grad = trainable or name.startswith("fc.")

def freeze_bn(model):
    '''Keep ImageNet BatchNorm statistics while the backbone is frozen.'''
    for m in model.modules():
        if isinstance(m, nn.BatchNorm2d):
            m.eval()

def train_one_epoch(model, loader, criterion, optimizer, scaler, backbone_frozen):
    model.train()
    if backbone_frozen:
        freeze_bn(model)

    loss_sum, correct, seen = 0.0, 0, 0
    for x, y in loader:
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device.type, enabled=use_amp):
            logits = model(x)
            loss = criterion(logits, y)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        loss_sum += loss.item() * x.size(0)
        correct += (logits.argmax(1) == y).sum().item()
        seen += x.size(0)

    return loss_sum / seen, correct / seen

@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    loss_sum, correct, seen = 0.0, 0, 0
    preds, targets = [], []

    for x, y in loader:
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        with torch.autocast(device.type, enabled=use_amp):
            logits = model(x)
            loss = criterion(logits, y)

        loss_sum += loss.item() * x.size(0)
        pred = logits.argmax(1)
        correct += (pred == y).sum().item()
        seen += x.size(0)
        preds.append(pred.cpu())
        targets.append(y.cpu())

    return (loss_sum / seen, correct / seen,
            torch.cat(preds).numpy(), torch.cat(targets).numpy())

In [ ]:
history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": [], "lr": []}
best_val_acc = -1.0
ckpt_path = Path("../checkpoints/resnet18_best.pt")
ckpt_path.parent.mkdir(parents=True, exist_ok=True)

backbone_frozen = FREEZE_BACKBONE

for epoch in range(EPOCHS):
    # optionally unfreeze the backbone for fine-tuning
    if (FREEZE_BACKBONE and backbone_frozen
            and UNFREEZE_AT is not None and epoch == UNFREEZE_AT):
        set_backbone_trainable(model, True)
        backbone_frozen = False
        optimizer = make_optimizer(model)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer, T_max=max(1, EPOCHS - epoch))
        print(f"[epoch {epoch}] backbone unfrozen -> fine-tuning")

    t0 = time.time()
    tr_loss, tr_acc = train_one_epoch(model, train_loader, criterion, optimizer,
                                      scaler, backbone_frozen)
    va_loss, va_acc, _, _ = evaluate(model, val_loader, criterion)
    scheduler.step()

    history["train_loss"].append(tr_loss)
    history["train_acc"].append(tr_acc)
    history["val_loss"].append(va_loss)
    history["val_acc"].append(va_acc)
    history["lr"].append(optimizer.param_groups[0]["lr"])

    print(f"epoch {epoch + 1:02d}/{EPOCHS} | "
          f"train loss {tr_loss:.4f} acc {tr_acc:.3f} | "
          f"val loss {va_loss:.4f} acc {va_acc:.3f} | "
          f"{time.time() - t0:.0f}s")

    if va_acc > best_val_acc:
        best_val_acc = va_acc
        torch.save({"model": model.state_dict(), "classes": classes,
                    "img_size": IMG_SIZE, "val_acc": va_acc}, ckpt_path)
        print(f"  -> new best saved ({va_acc:.3f})")

print(f"best val accuracy: {best_val_acc:.3f}")

## 5. Training curves

In [ ]:
epochs = range(1, len(history["train_loss"]) + 1)
best_epoch = int(np.argmax(history["val_acc"])) + 1

fig, axes = plt.subplots(1, 3, figsize=(16, 4.2))

axes[0].plot(epochs, history["train_loss"], marker="o", label="train")
axes[0].plot(epochs, history["val_loss"], marker="o", label="val")
axes[0].set(title="Loss", xlabel="epoch", ylabel="cross-entropy")
axes[0].legend()

axes[1].plot(epochs, history["train_acc"], marker="o", label="train")
axes[1].plot(epochs, history["val_acc"], marker="o", label="val")
axes[1].axvline(best_epoch, color="grey", ls="--", alpha=0.6, label=f"best ({best_epoch})")
axes[1].set(title="Accuracy", xlabel="epoch", ylabel="accuracy")
axes[1].legend()

axes[2].plot(epochs, history["lr"], marker="o", color="tab:green")
axes[2].set(title="Learning rate", xlabel="epoch", ylabel="lr")
axes[2].set_yscale("log")

for ax in axes:
    ax.grid(alpha=0.3)

fig.suptitle("ResNet-18 transfer learning", y=1.03)
plt.tight_layout()
plt.show()

## 6. Evaluate the best checkpoint

If CIFAR-10 is used, this is the official test split. For your own folder there is no separate test set, so the validation split is reused — treat those numbers as validation, not test.

In [ ]:
checkpoint = torch.load(ckpt_path, map_location=device, weights_only=True)
model.load_state_dict(checkpoint["model"])
model.eval()
print(f"loaded best checkpoint (val acc {checkpoint['val_acc']:.3f})")

eval_name = "test" if test_loader is not None else "val"
eval_loader = test_loader if test_loader is not None else val_loader

test_loss, test_acc, y_pred, y_true = evaluate(
    model, eval_loader, nn.CrossEntropyLoss())
print(f"{eval_name}: loss {test_loss:.4f} | accuracy {test_acc:.3f}")

print("\n" + classification_report(y_true, y_pred, labels=range(len(classes)),
                                   target_names=classes, digits=3, zero_division=0))

In [ ]:
cm = confusion_matrix(y_true, y_pred, labels=range(len(classes)))
cm_norm = cm / np.maximum(cm.sum(axis=1, keepdims=True), 1)

fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(cm_norm, cmap="Blues", vmin=0, vmax=1)
ax.set(title=f"Confusion matrix ({eval_name})",
       xlabel="predicted", ylabel="true")

if len(classes) <= 20:
    ax.set_xticks(range(len(classes)))
    ax.set_yticks(range(len(classes)))
    ax.set_xticklabels(classes, rotation=45, ha="right")
    ax.set_yticklabels(classes)

fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label="row-normalised")
plt.tight_layout()
plt.show()

In [ ]:
@torch.no_grad()
def show_predictions(model, loader, n=12):
    model.eval()
    images, labels = next(iter(loader))
    images = images[:n]
    preds = model(images.to(device)).argmax(1).cpu()

    cols = 6
    rows = int(np.ceil(len(images) / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(3 * cols, 3 * rows))

    for ax, img, true, pred in zip(axes.ravel(), images, labels, preds):
        ax.imshow(denormalize(img).permute(1, 2, 0))
        ok = int(true) == int(pred)
        ax.set_title(f"true: {classes[true]}\npred: {classes[pred]}",
                     fontsize=9, color="green" if ok else "crimson")
        ax.axis("off")

    for ax in axes.ravel()[len(images):]:
        ax.axis("off")

    plt.tight_layout()
    plt.show()

show_predictions(model, eval_loader)

## 7. Going further

**Reload the model elsewhere** — the last cell defines `load_model(path)`, which rebuilds the architecture and restores the saved weights.

**512-d penultimate features** (for linear probes, clustering, weight-space experiments)

```python
features, targets = extract_features(model, val_loader)   # (N, 512), (N,)
```

**Tips**

* CPU only? Set `IMG_SIZE = 128`, `TRAIN_FRACTION = 0.2`, `EPOCHS = 2` first.
* Underfitting? Unfreeze earlier (`UNFREEZE_AT = 1`), train longer, or raise `LR_BACKBONE` to `3e-4`.
* Overfitting? Keep the backbone frozen, increase `WEIGHT_DECAY`, add augmentation.
* Class imbalance? Use `WeightedRandomSampler` or pass class weights to `CrossEntropyLoss`.
* Compare with from-scratch training by using `models.resnet18(weights=None)`.
* To resume training, also save `optimizer.state_dict()`, `scheduler.state_dict()` and the epoch.

In [ ]:
# 512-d penultimate features (useful for linear probes / weight-space experiments)
@torch.no_grad()
def extract_features(model, loader):
    backbone = nn.Sequential(*list(model.children())[:-1]).to(device).eval()
    feats, ys = [], []
    for x, y in loader:
        feats.append(backbone(x.to(device)).flatten(1).cpu())
        ys.append(y)
    return torch.cat(feats), torch.cat(ys)

features, targets = extract_features(model, val_loader)
print("features:", tuple(features.shape), "| targets:", tuple(targets.shape))

In [ ]:
# Weight-space view: flatten every parameter into a single vector
def state_dict_to_vector(model):
    return torch.cat([p.detach().flatten().cpu() for p in model.parameters()])

w = state_dict_to_vector(model)
print("weight vector:", tuple(w.shape))

In [ ]:
def load_model(path, device=device):
    ckpt = torch.load(path, map_location=device, weights_only=True)
    m = build_model(len(ckpt["classes"]), freeze_backbone=False)
    m.load_state_dict(ckpt["model"])
    return m.to(device).eval(), ckpt["classes"]

# model2, classes2 = load_model("../checkpoints/resnet18_best.pt")